# 12. UOL Classifiers for Evolving Feature Spaces

OpenMOA provides **10 classifiers** designed for **Utilitarian Online Learning (UOL)** —
the setting where the feature space itself changes over time.  All classifiers share
a common interface:

```python
learner.train(instance)            # update model on one instance
learner.predict(instance)          # → class index
learner.predict_proba(instance)    # → probability vector
```

All classifiers are compatible with the variable-length output of `OpenFeatureStream`
via the `feature_indices` mechanism (see notebook 11).

| # | Algorithm | Key idea | Multi-class | PyTorch |
|---|-----------|----------|-------------|---------|
| 1 | **FESL** | Sparse weights + Ridge mapping between spaces | Binary only | No |
| 2 | **OASF** | Passive-Aggressive + L₁,₂ group sparsity | Binary only | No |
| 3 | **RSOL** | Random projection + sparse online update | Binary only | No |
| 4 | **FOBOS** | Proximal gradient with L1/L2 regularisation | Binary + Multi | No |
| 5 | **FTRL** | Per-coordinate adaptive learning rates | Binary + Multi | No |
| 6 | **ORF3V** | Online Random Forest on variable-length vectors | Binary + Multi | No |
| 7 | **OVFM** | Variational Factorisation Machine + Gaussian Copula | Binary only | Yes |
| 8 | **OSLMF** | DensityPeaks + Copula + semi-supervised MF | Binary only | Yes |
| 9 | **OLD3S** | Deep VAE + hierarchical Bayesian pruning | Binary + Multi | Yes |
|10 | **OWSS** | Importance-weighted sparse online learning | Binary + Multi | Yes |

In [ ]:
# This cell is hidden on openmoa.net. See docs/contributing/docs.rst
from util.nbmock import mock_datasets, is_nb_fast
if is_nb_fast():
    mock_datasets()

In [ ]:
import os
from collections import deque

import numpy as np
import matplotlib.pyplot as plt

from openmoa.datasets import Electricity, ElectricityTiny
from openmoa.stream.stream_wrapper import OpenFeatureStream, ShuffledStream

# Non-PyTorch classifiers
from openmoa.classifier._fesl_classifier  import FESLClassifier
from openmoa.classifier._oasf_classifier  import OASFClassifier
from openmoa.classifier._rsol_classifier  import RSOLClassifier
from openmoa.classifier._fobos_classifier import FOBOSClassifier
from openmoa.classifier._ftrl_classifier  import FTRLClassifier
from openmoa.classifier._orf3v_classifier import ORF3VClassifier

# PyTorch classifiers
from openmoa.classifier._ovfm_classifier  import OVFMClassifier
from openmoa.classifier._oslmf_classifier import OSLMFClassifier
from openmoa.classifier._old3s_classifier import OLD3SClassifier
from openmoa.classifier._owss_classifier  import OWSSClassifier

NB_FAST    = os.environ.get("NB_FAST", "False").lower() == "true"
DatasetCls = ElectricityTiny if NB_FAST else Electricity
N_TOTAL    = 1_000 if NB_FAST else 10_000
WINDOW     = 200   if NB_FAST else 1_000   # prequential window size

print(f"Dataset  : {DatasetCls.__name__}  ({N_TOTAL} instances)")
print(f"Window   : {WINDOW}")

---
## 1. Prequential Evaluation

All UOL experiments use the **prequential (test-then-train)** protocol:
each instance is first used to make a prediction, then used to update the model.
No separate test set is needed.

We track two metrics:
- **Prequential accuracy** — sliding-window accuracy over the last `W` instances
- **Cumulative accuracy** — overall accuracy from instance 1 to now

In [ ]:
def prequential_eval(stream, learner, n_instances: int, window_size: int = 1_000):
    """Run a prequential (test-then-train) evaluation loop.

    Parameters
    ----------
    stream      : any OpenMOA stream (wrapper or base)
    learner     : any UOL classifier with .predict() and .train()
    n_instances : number of instances to process
    window_size : sliding window length for prequential accuracy

    Returns
    -------
    steps      : list of int   — step indices at each log point
    preq_accs  : list of float — prequential accuracy
    cum_accs   : list of float — cumulative accuracy
    """
    window        = deque(maxlen=window_size)
    total_correct = 0
    steps, preq_accs, cum_accs = [], [], []
    log_every = max(1, n_instances // 100)

    for t in range(1, n_instances + 1):
        if not stream.has_more_instances():
            break
        inst = stream.next_instance()

        pred          = learner.predict(inst)
        correct       = int(pred == inst.y_index)
        total_correct += correct
        window.append(correct)

        learner.train(inst)

        if t % log_every == 0 or t == n_instances:
            steps.append(t)
            preq_accs.append(sum(window) / len(window))
            cum_accs.append(total_correct / t)

    return steps, preq_accs, cum_accs


def make_stream(seed=1, pattern="eds", **kw):
    """Convenience factory: ShuffledStream → OpenFeatureStream."""
    defaults = dict(n_segments=3, overlap_ratio=0.2)
    defaults.update(kw)
    return OpenFeatureStream(
        base_stream=ShuffledStream(DatasetCls(), random_seed=seed),
        evolution_pattern=pattern,
        total_instances=N_TOTAL,
        random_seed=seed,
        **defaults,
    )


# Shared schema (all streams over Electricity have the same schema)
_ref   = make_stream(seed=1)
schema = _ref.get_schema()
d      = schema.get_num_attributes()
print(f"Schema: {d} features, {schema.get_num_classes()} classes")

---
## 2. Single-Algorithm Walkthrough: FESL

**FESL (Feature Evolvable Streaming Learning)** — Hou et al., NeurIPS 2017.

Maintains two sparse linear models keyed by global feature ID:
- `w_curr` — weights for the current feature space
- `w_old`  — weights for the previous feature space

When a feature-space shift is detected via Jaccard distance, FESL learns a Ridge
Regression mapping `M : S_new → S_old` to transfer knowledge between spaces.
Prediction is a weighted ensemble of both models.

In [ ]:
stream  = make_stream(seed=1)
learner = FESLClassifier(
    schema=schema,
    alpha=0.1,        # gradient step size
    lambda_=0.1,      # Ridge regularisation on the mapping matrix
    window_size=100,  # buffer size for learning the mapping
    random_seed=1,
)

steps, preq_accs, cum_accs = prequential_eval(stream, learner, N_TOTAL, WINDOW)
print(f"FESL | prequential: {preq_accs[-1]:.3f} | cumulative: {cum_accs[-1]:.3f}")

plt.figure(figsize=(10, 3))
plt.plot(steps, preq_accs, lw=2, label=f"Prequential (w={WINDOW})")
plt.plot(steps, cum_accs,  lw=2, ls="--", label="Cumulative")
plt.xlabel("Instances processed"); plt.ylabel("Accuracy")
plt.title("FESL on Electricity — EDS (3 segments, overlap=0.2)")
plt.ylim(0, 1.05); plt.legend(); plt.grid(ls="--", alpha=0.5)
plt.tight_layout(); plt.show()

---
## 3. Three Dynamic Feature-Space Paradigms

The OpenMOA benchmark evaluates every algorithm under three paradigms:

| Paradigm | Description |
|----------|-------------|
| **TDS** | Features enter in 10 ordered birth stages |
| **CDS** | Each feature independently missing with probability 0.4 |
| **EDS** | Feature space rotates through 3 segments with 20% overlap |

In [ ]:
paradigms = {
    "TDS": dict(pattern="tds", tds_mode="ordered"),
    "CDS": dict(pattern="cds", missing_ratio=0.4),
    "EDS": dict(pattern="eds", n_segments=3, overlap_ratio=0.2),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), sharey=True)
for ax, (name, kw) in zip(axes, paradigms.items()):
    stream  = make_stream(seed=1, **kw)
    learner = FESLClassifier(schema=schema, alpha=0.1, random_seed=1)
    steps, preq, cum = prequential_eval(stream, learner, N_TOTAL, WINDOW)
    ax.plot(steps, preq, lw=2,          label=f"Prequential (w={WINDOW})")
    ax.plot(steps, cum,  lw=2, ls="--", label="Cumulative")
    ax.set_title(f"FESL — {name}", fontweight="bold")
    ax.set_xlabel("Instances"); ax.set_ylim(0, 1.05); ax.grid(ls="--", alpha=0.4)
    if ax is axes[0]:
        ax.set_ylabel("Accuracy"); ax.legend(fontsize=9)

plt.suptitle("FESL across TDS / CDS / EDS paradigms", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

---
## 4. All 10 Classifiers — Side-by-Side Comparison

We evaluate all 10 UOL classifiers on the **EDS** paradigm.
PyTorch classifiers (OVFM, OSLMF, OLD3S, OWSS) run on CPU by default;
they automatically use CUDA if available.

In [ ]:
classifiers = {
    # ── Non-PyTorch ──────────────────────────────────────────────────────
    "FESL":  FESLClassifier(
                schema=schema, alpha=0.1, lambda_=0.1, window_size=100, random_seed=1),
    "OASF":  OASFClassifier(
                schema=schema, lambda_param=0.01, mu=1.0, L=100, random_seed=1),
    "RSOL":  RSOLClassifier(
                schema=schema, lambda_param=0.01, mu=1.0, L=100, random_seed=1),
    "FOBOS": FOBOSClassifier(
                schema=schema, alpha=1.0, lambda_=0.001,
                regularization="l1", random_seed=1),
    "FTRL":  FTRLClassifier(
                schema=schema, alpha=0.1, beta=1.0, l1=0.1, l2=0.0, random_seed=1),
    "ORF3V": ORF3VClassifier(
                schema=schema, n_stumps=10, alpha=0.1, d_max=d, random_seed=1),
    # ── PyTorch ──────────────────────────────────────────────────────────
    "OVFM":  OVFMClassifier(
                schema=schema, window_size=200, batch_size=50,
                learning_rate=0.01, l2_lambda=0.01, random_seed=1),
    "OSLMF": OSLMFClassifier(
                schema=schema, window_size=200, buffer_size=200,
                batch_size=50, learning_rate=0.01, random_seed=1),
    "OLD3S": OLD3SClassifier(
                schema=schema, latent_dim=20, hidden_dim=128,
                num_hbp_layers=3, learning_rate=0.001, random_seed=1),
    "OWSS":  OWSSClassifier(
                schema=schema, window_size=100, hidden_dim=32,
                learning_rate=0.01, rec_weight=0.1, random_seed=1),
}

results = {}
for name, learner in classifiers.items():
    stream = make_stream(seed=1)
    steps, preq, cum = prequential_eval(stream, learner, N_TOTAL, WINDOW)
    results[name] = (steps, preq, cum)
    print(f"{name:<8} | preq: {preq[-1]:.3f} | cum: {cum[-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
colors = plt.cm.tab10.colors
linestyles = ["-"] * 6 + ["--"] * 4   # solid = no-PyTorch, dashed = PyTorch

for i, (name, (steps, preq, cum)) in enumerate(results.items()):
    c  = colors[i % 10]
    ls = linestyles[i]
    axes[0].plot(steps, preq, lw=2, color=c, ls=ls, label=name)
    axes[1].plot(steps, cum,  lw=2, color=c, ls=ls, label=name)

for ax, title in zip(axes, [
    f"Prequential Accuracy (window={WINDOW})",
    "Cumulative Accuracy",
]):
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Instances processed")
    ax.set_ylabel("Accuracy")
    ax.set_ylim(0, 1.05)
    ax.legend(ncol=2, fontsize=9)
    ax.grid(ls="--", alpha=0.4)

fig.suptitle(
    "All 10 UOL classifiers on Electricity — EDS paradigm\n"
    "(solid = no PyTorch, dashed = PyTorch)",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.show()

---
## 5. Per-Algorithm Parameter Guide

### Non-PyTorch

**FESL** — best for abrupt space shifts with a learnable mapping structure.
```python
FESLClassifier(schema, alpha=0.1, lambda_=0.1, window_size=100)
# alpha      : gradient step size
# lambda_    : Ridge penalty on the mapping matrix M
# window_size: how many overlap instances to use when fitting M
```

**OASF / RSOL** — best for high-dimensional sparse streams (e.g. RCV1 ~47k features).
```python
OASFClassifier(schema, lambda_param=0.01, mu=1.0, L=100)
RSOLClassifier(schema, lambda_param=0.01, mu=1.0, L=100)
# lambda_param: L1,2 group sparsity penalty
# mu          : passive-aggressive aggressiveness
# L           : sliding window length
```

**FOBOS** — general-purpose; supports binary and multi-class.
```python
FOBOSClassifier(schema, alpha=1.0, lambda_=0.001,
                regularization="l1",      # or "l2", "l1_l2", "none"
                step_schedule="sqrt")     # or "linear"
```

**FTRL** — general-purpose; per-feature adaptive learning rates.
```python
FTRLClassifier(schema, alpha=0.1, beta=1.0, l1=0.1, l2=0.0)
# alpha/beta: AdaGrad-style learning rate schedule
# l1/l2     : regularisation strengths
```

**ORF3V** — non-linear; handles arbitrary feature-ID spaces via per-feature forests.
```python
ORF3VClassifier(schema, n_stumps=10, alpha=0.1, d_max=d,
                replacement_interval=200, replacement_strategy="worst")
# n_stumps            : stumps per feature forest
# d_max               : maximum expected global feature ID + 1
# replacement_interval: how often to prune underperforming stumps
```

### PyTorch

**OVFM** — Gaussian Copula + Factorisation Machine; best for NaN-filling streams (CDS/TDS).
```python
OVFMClassifier(schema, window_size=200, batch_size=50,
               learning_rate=0.01, l2_lambda=0.01,
               decay_coef=0.5)   # Copula covariance decay
```

**OSLMF** — DensityPeaks + Copula; semi-supervised (works with unlabelled instances).
```python
OSLMFClassifier(schema, window_size=200, buffer_size=200,
                batch_size=50,    # DensityPeaks runs once per batch
                learning_rate=0.01, l2_lambda=0.001)
```

**OLD3S** — Deep VAE + hierarchical Bayesian pruning; supports knowledge distillation across space shifts.
```python
OLD3SClassifier(schema, latent_dim=20, hidden_dim=128,
                num_hbp_layers=3, learning_rate=0.001,
                beta=0.99,   # HBP weight decay rate
                eta=0.01)    # ensemble update rate
```

**OWSS** — Autoencoder + classifier; handles open-world features via weight expansion.
```python
OWSSClassifier(schema, window_size=100, hidden_dim=32,
               learning_rate=0.01,
               rec_weight=0.1)   # reconstruction loss weight (Eq. 2)
```

---
## 6. `predict_proba` — Probability Output

All 10 classifiers expose `predict_proba(instance)` returning a probability
vector over classes.  The vector sums to 1 and each value lies in [0, 1].

In [ ]:
# Warm up each classifier on 200 instances, then check predict_proba
warmup_n = 200

print(f"{'Classifier':<10} {'P(class=0)':<12} {'P(class=1)':<12} {'Sum':<8} {'Valid'}")
print("-" * 55)

for name, learner in classifiers.items():
    stream = make_stream(seed=42)
    for _ in range(warmup_n):
        inst = stream.next_instance()
        learner.train(inst)
    inst  = stream.next_instance()
    proba = learner.predict_proba(inst)
    total = sum(proba)
    valid = abs(total - 1.0) < 1e-4 and all(0 <= p <= 1 for p in proba)
    print(f"{name:<10} {proba[0]:<12.4f} {proba[1]:<12.4f} {total:<8.4f} {'✓' if valid else '✗'}")

---
## 7. Reproducibility

All classifiers use instance-level RNGs (`np.random.RandomState` for NumPy,
`torch.manual_seed` for PyTorch models).  Two runs with the same `random_seed`
on the same stream produce **bit-identical** predictions.

In [ ]:
n_check = 50

def collect_preds(LearnerCls, kwargs, seed):
    stream  = make_stream(seed=seed)
    learner = LearnerCls(schema=schema, random_seed=seed, **kwargs)
    preds = []
    for _ in range(n_check):
        inst = stream.next_instance()
        preds.append(learner.predict(inst))
        learner.train(inst)
    return preds

check_pairs = [
    ("FESL",  FESLClassifier,  dict(alpha=0.1)),
    ("FOBOS", FOBOSClassifier, dict(alpha=1.0)),
    ("OVFM",  OVFMClassifier,  dict()),
    ("OWSS",  OWSSClassifier,  dict()),
]

for name, cls, kw in check_pairs:
    run_a = collect_preds(cls, kw, seed=7)
    run_b = collect_preds(cls, kw, seed=7)
    run_c = collect_preds(cls, kw, seed=99)
    same  = run_a == run_b
    diff  = run_a != run_c
    print(f"{name:<8} same-seed identical: {'✓' if same else '✗'}  "
          f"diff-seed differs: {'✓' if diff else '✗'}")

---
## Summary

| Algorithm | Best suited for | Key parameters |
|-----------|----------------|----------------|
| **FESL**  | Abrupt space shifts with mappable structure | `alpha`, `window_size` |
| **OASF**  | High-dimensional sparse streams (incl. RCV1) | `lambda_param`, `L` |
| **RSOL**  | High-dimensional sparse streams | `lambda_param`, `L` |
| **FOBOS** | General-purpose binary/multi-class | `alpha`, `regularization` |
| **FTRL**  | General-purpose; per-feature adaptive LR | `alpha`, `l1`, `l2` |
| **ORF3V** | Non-linear, open feature-ID spaces | `n_stumps`, `d_max` |
| **OVFM**  | NaN-filling streams (CDS/TDS) with mixed feature types | `window_size`, `decay_coef` |
| **OSLMF** | Semi-supervised; partial labels | `batch_size`, `buffer_size` |
| **OLD3S** | Deep latent-space alignment across multiple space shifts | `latent_dim`, `num_hbp_layers` |
| **OWSS**  | Open-world features; reconstruction-regularised classification | `hidden_dim`, `rec_weight` |

All classifiers:
- Accept instances from any OpenMOA stream wrapper
- Maintain weights keyed by **global feature ID** to correctly handle index shifts
- Are reproducible via `random_seed`
- Expose `predict_proba()` returning a calibrated probability vector